# 03 — Build the component and client tables

Assemble one row per Liu gene without silently choosing among conflicting current annotations. Original Liu columns remain in the Notebook 01 interim files; processed outputs use the controlled atlas schema.

In [1]:
import json, os, sys
from pathlib import Path

import pandas as pd
import yaml

sys.path.insert(0, os.path.abspath(".."))
from atlas import crosswalk
from atlas.schema import (
    COLUMN_ORDER, DATA_DICTIONARY, REQUIRED_FIELDS, SUBSYSTEM_ORDER, EvidenceSource,
    GlycosylationRole, MappingStatus, RecordOrigin, RecordType,
)
Path("../data/processed").mkdir(parents=True, exist_ok=True)
Path("../data/processed/qa").mkdir(parents=True, exist_ok=True)
with open("../config/sources.yaml", encoding="utf-8") as handle:
    cfg = yaml.safe_load(handle)

## Purpose: prepare current annotations without arbitrary candidate selection

In [2]:
resolved = pd.read_parquet("../data/interim/seed_resolved.parquet")
uniprot_locus = pd.read_parquet("../data/interim/uniprot_locus_crosswalk.parquet")
clients_source = pd.read_csv("../data/interim/liu_clients_raw_cleaned.csv")
pathway_members = pd.read_parquet("../data/interim/kegg_pathways.parquet")
kegg_crosswalk = pd.read_parquet("../data/interim/crosswalk.parquet")
assert len(resolved) == 369 and len(clients_source) == 2269

candidate_counts = (uniprot_locus.groupby("ao_locus_tag")["uniprot_accession"]
                    .nunique().rename("annotation_candidate_count"))
single = (uniprot_locus.sort_values(["ao_locus_tag", "uniprot_accession"])
          .drop_duplicates("ao_locus_tag")
          .merge(candidate_counts, on="ao_locus_tag", how="left"))
# Values from multi-candidate loci are deliberately withheld from the final
# gene row. Every candidate remains available in identifier_crosswalk.csv.
annotation_fields = ["kegg_gene_id", "uniprot_accession", "ncbi_gene_id",
                     "gene_name", "function", "compartment_raw"]
single.loc[single.annotation_candidate_count > 1, annotation_fields] = None
single = single.rename(columns={c: f"annotation_{c}" for c in annotation_fields})
print("annotation candidates per locus:", candidate_counts.value_counts().sort_index().to_dict())

annotation candidates per locus: {1: 12059, 2: 5}


## Machinery: normalize observed Liu categories and attach annotations

In [3]:
SUBSYSTEM_MAP = {
    "TC": "tc", "Dolichol pathway": "dolichol_pathway",
    "Erglycosylation": "er_glycosylation", "Folding": "folding",
    "GPI biosynthesis": "gpi_biosynthesis", "ERAD": "erad",
    "COPII": "copii", "COPI": "copi",
    "Golgi processing": "golgi_processing", "LDSV": "ldsv",
    "HDSV": "hdsv", "CPY pathway": "cpy_pathway",
    "ALPpathway": "alp_pathway", "SNARE": "snare",
    "Septin": "septin",
    "beta-1,6 glucan biosynthesis": "beta_1_6_glucan_biosynthesis",
    "Translation": "translation",
    "putative mitochondria protein": "putative_mitochondria_protein",
    "mitochondrial m‐AAA protease": "mitochondrial_m_aaa_protease",
}
machinery = resolved.merge(single, on="ao_locus_tag", how="left", validate="many_to_one")
assert len(machinery) == 369 and machinery.record_id.is_unique
machinery["gene_name"] = machinery["annotation_gene_name"]
machinery["function"] = machinery["Description"].fillna(machinery["annotation_function"])
machinery["uniprot_accession"] = machinery["annotation_uniprot_accession"]
machinery["kegg_gene_id"] = machinery["annotation_kegg_gene_id"]
machinery["ncbi_gene_id"] = machinery["annotation_ncbi_gene_id"]
ko_lookup = (kegg_crosswalk.dropna(subset=["kegg_ko"]).drop_duplicates("ao_locus_tag").set_index("ao_locus_tag")["kegg_ko"])
machinery["kegg_ko"] = machinery.ao_locus_tag.map(ko_lookup)
machinery["yeast_ortholog"] = machinery["S. cerevisiae ortholog"]
source_columns = ["1st SOURCE", "2nd SOURCE", "3rd SOURCE", "4th SOURCE"]
machinery["liu_source_raw"] = machinery[source_columns].apply(lambda row: "|".join("" if pd.isna(value) else str(value) for value in row), axis=1)
machinery["liu_table_row"] = [f"S1:{row}" for row in range(3, 3 + len(machinery))]
machinery["subsystem"] = machinery["Subsystems or function"].map(SUBSYSTEM_MAP)
machinery["pathway_order"] = machinery["subsystem"].map(SUBSYSTEM_ORDER)
before_subsystem_coverage = int(machinery["subsystem"].notna().sum())
feizi = pd.read_excel("../data/reference/feizi2013_table_s1.xlsx", sheet_name="Sheet1", header=1)
FEIZI_MAP = {
    "Translocation": "tc", "Dolichol": "dolichol_pathway",
    "ERglycosylation": "er_glycosylation", "protein folding": "folding",
    "GPI biosynthesis": "gpi_biosynthesis", "ERADL": "erad",
    "ERADM": "erad", "ERADC": "erad", "COPII": "copii",
    "COPI": "copi", "Golgi processing": "golgi_processing",
    "LDSV": "ldsv", "HDSV": "hdsv", "CPY pathway": "cpy_pathway",
    "ALP pathway": "alp_pathway", "SNARE": "snare",
}
scaffold = feizi.rename(columns={"Standard Name": "yeast_ortholog"})[["yeast_ortholog", "SUBSYSTEM"]]
scaffold["subsystem"] = scaffold["SUBSYSTEM"].map(FEIZI_MAP)
scaffold = scaffold.dropna(subset=["subsystem"])
machinery = crosswalk.assign_subsystems_from_evidence(machinery, scaffold)
CURATED_SUBSYSTEMS = {
    "AO090701000141": {"subsystem": "er_glycosylation", "source": "liu2014_description+uniprot+kegg_ko", "confidence": "high", "rationale": "Glucosidase I trims the first glucose from N-glycans in the ER; folding and ER-processing memberships are secondary context."},
    "AO090003001225": {"subsystem": "er_glycosylation", "source": "liu2014_description+uniprot+kegg_ko", "confidence": "high", "rationale": "Class I alpha-mannosidase (KO K01230) performs ER N-glycan mannose trimming; ER quality control is a secondary role."},
    "AO090020000468": {"subsystem": "er_glycosylation", "source": "liu2014_description+uniprot+kegg_ko", "confidence": "high", "rationale": "WBP1 is the ER oligosaccharyltransferase beta subunit (KO K12670), making N-glycan transfer its most specific function."},
    "AO090009000178": {"subsystem": "erad", "source": "aspergillus_homolog+uniprot+yeast_ortholog+kegg_ko", "confidence": "medium", "rationale": "The A. oryzae protein is a class I alpha-mannosidase (KO K01230); its MNL2 ortholog is ER-localized and functions in ERAD quality control."},
    "AO090003000257": {"subsystem": "folding", "source": "curated_liu_composite", "confidence": "high", "rationale": "KAR2/BiP is the primary ER Hsp70 folding chaperone; its ERAD contribution is secondary quality-control context.", "override_verified_liu": True},
}
machinery = crosswalk.apply_curated_subsystem_resolutions(machinery, CURATED_SUBSYSTEMS)
machinery = crosswalk.assign_missing_subsystems(
    machinery, pathway_members, cfg["kegg_pathway_to_subsystem"]
)
source_confidence = {"liu2014": "high", "liu2014_description": "medium", "yeast_scaffold": "medium", "current_annotation": "medium", "kegg_pathway": "low", "unassigned": "low"}
machinery["subsystem_confidence"] = machinery["subsystem_confidence"].fillna(machinery.subsystem_source.map(source_confidence))
default_rationale = machinery.subsystem_source.map({"liu2014": "Verified Liu 2014 Table S1 subsystem.", "liu2014_description": "Specific function stated in Liu 2014 description.", "yeast_scaffold": "Unique match in the original yeast secretory-system scaffold.", "current_annotation": "Specific current database function annotation.", "kegg_pathway": "Single mapped KEGG pathway; broad, low-confidence placement.", "unassigned": "Available evidence does not distinguish a primary subsystem."})
machinery["subsystem_rationale"] = machinery["subsystem_rationale"].fillna(default_rationale)
after_subsystem_coverage = int(machinery["subsystem"].notna().sum())
print(f"subsystem coverage: {before_subsystem_coverage}/{len(machinery)} -> {after_subsystem_coverage}/{len(machinery)}")
print(machinery["subsystem_source"].value_counts(dropna=False))
machinery = crosswalk.assign_controlled_compartments(machinery)
machinery["citation"] = "10.1186/1752-0509-8-73"
machinery["record_type"] = RecordType.MACHINERY.value
machinery["record_origin"] = RecordOrigin.LIU2014.value

subsystem coverage: 109/369 -> 247/369
subsystem_source
unassigned                                            122
liu2014                                               108
liu2014_description                                    74
yeast_scaffold                                         44
kegg_pathway                                           13
current_annotation                                      3
liu2014_description+uniprot+kegg_ko                     3
curated_liu_composite                                   1
aspergillus_homolog+uniprot+yeast_ortholog+kegg_ko      1
Name: count, dtype: int64


In [4]:
# Functional-assignment evidence is not upgraded merely because expression
# changed. Transcriptomics supports response, not the inferred function.
source_text = machinery[["1st SOURCE", "2nd SOURCE", "3rd SOURCE", "4th SOURCE"]].fillna("").agg(" | ".join, axis=1)
machinery["evidence_source"] = EvidenceSource.UNKNOWN.value
machinery.loc[source_text.str.contains("inparanoid|psi-blast", case=False), "evidence_source"] = EvidenceSource.YEAST_INFERENCE.value
machinery.loc[source_text.str.contains("Oliveira", case=False), "evidence_source"] = EvidenceSource.ASPERGILLUS_HOMOLOG.value

multi = machinery.annotation_candidate_count.gt(1)
machinery.loc[multi, "mapping_status"] = MappingStatus.AMBIGUOUS.value
machinery.loc[multi, "mapping_method"] = "AO090 tag has multiple UniProt candidates; no candidate selected"
machinery["manual_review_required"] = machinery["manual_review_required"].fillna(False) | multi
machinery["manual_review_required"] |= machinery.mapping_status.isin(["unresolved", "ambiguous", "split", "merged"])

# Conservative role rules always retain their provenance and confidence.
machinery = crosswalk.assign_glycosylation_roles(machinery, cfg["glycosylation_ko"])
trimming_tags = ["AO090701000141", "AO090003001225", "AO090009000178"]
machinery.loc[machinery.liu_ao_locus_tag.isin(trimming_tags), ["glycosylation_role", "glycosylation_role_source", "glycosylation_role_confidence"]] = ["n_glycan_trimming", "curated_multi_source_evidence", "high"]
machinery.loc[machinery.liu_ao_locus_tag.eq("AO090020000468"), ["glycosylation_role", "glycosylation_role_source", "glycosylation_role_confidence"]] = ["n_glycan_transfer", "curated_multi_source_evidence", "high"]
print("glycosylation roles:", machinery.glycosylation_role.value_counts().to_dict())

glycosylation roles: {'none': 307, 'gpi_anchor': 20, 'n_glycan_assembly': 14, 'golgi_mannosylation': 12, 'n_glycan_transfer': 9, 'n_glycan_trimming': 7}


## Clients: preserve the FSD population as a separate table

In [5]:
clients = clients_source.rename(columns={"ID": "liu_ao_locus_tag"})
clients = crosswalk.make_record_ids(clients, prefix="AOC")
clients = clients.merge(single, left_on="liu_ao_locus_tag", right_on="ao_locus_tag", how="left", validate="many_to_one")
clients["annotation_candidate_count"] = clients["annotation_candidate_count"].fillna(0).astype(int)
clients["gene_name"] = clients["annotation_gene_name"]
clients["function"] = clients["annotation_function"].fillna(clients["Annotation"])
clients["uniprot_accession"] = clients["annotation_uniprot_accession"]
clients["kegg_gene_id"] = clients["annotation_kegg_gene_id"]
clients["ncbi_gene_id"] = clients["annotation_ncbi_gene_id"]
clients["kegg_ko"] = clients.ao_locus_tag.map(ko_lookup)
clients["record_type"] = RecordType.CLIENT.value
clients["record_origin"] = RecordOrigin.LIU2014.value
clients["evidence_source"] = EvidenceSource.DATABASE_PREDICTION.value
clients["citation"] = "10.1186/1752-0509-8-73"
clients["subsystem"] = None
clients["subsystem_source"] = "unassigned"
clients["subsystem_confidence"] = "low"
clients["subsystem_rationale"] = "Secretory clients are not assigned to machinery subsystems."
clients["pathway_order"] = None
clients["compartment"] = "unknown"
clients["compartment_source"] = "unassigned"
clients["compartment_confidence"] = "low"
clients["glycosylation_role"] = GlycosylationRole.NONE.value
clients["glycosylation_role_source"] = None
clients["glycosylation_role_confidence"] = None
clients["yeast_ortholog"] = None
clients["liu_source_raw"] = None
clients["liu_table_row"] = [f"S3:{row}" for row in range(3, 3 + len(clients))]
clients["mapping_status"] = MappingStatus.UNRESOLVED.value
clients["mapping_method"] = "no current AO090 cross-reference found"
clients["manual_review_required"] = False
one = clients.annotation_candidate_count.eq(1)
many = clients.annotation_candidate_count.gt(1)
clients.loc[one, "mapping_status"] = MappingStatus.EXACT.value
clients.loc[one, "mapping_method"] = "direct AO090 match in UniProt cross-reference"
clients.loc[many, "mapping_status"] = MappingStatus.AMBIGUOUS.value
clients.loc[many, "mapping_method"] = "AO090 tag has multiple UniProt candidates; no candidate selected"
clients.loc[many, "manual_review_required"] = True
assert len(clients) == 2269 and clients.record_id.is_unique
print(clients.mapping_status.value_counts())

mapping_status
exact         2268
unresolved       1
Name: count, dtype: int64


## Validate and write processed deliverables

In [6]:
machinery_final = machinery.reindex(columns=COLUMN_ORDER)
clients_final = clients.reindex(columns=COLUMN_ORDER)
for label, frame, expected in (("machinery", machinery_final, 369), ("clients", clients_final, 2269)):
    assert len(frame) == expected
    assert frame.record_id.is_unique
    for field in REQUIRED_FIELDS:
        assert frame[field].notna().all(), f"{label}: null {field}"
assert int(machinery_final.sig_all_three.sum()) == 51
assert machinery_final.loc[machinery_final.sig_all_three, "direction_all_three"].value_counts().to_dict() == {"up": 48, "down": 3}

machinery_final.to_csv("../data/processed/aoryzae_secretory_components.csv", index=False)
clients_final.to_csv("../data/processed/aoryzae_secretory_clients.csv", index=False)
uniprot_locus.to_csv("../data/processed/identifier_crosswalk.csv", index=False)
shortlist = machinery_final.loc[machinery_final.sig_all_three, ["ao_locus_tag", "gene_name", "yeast_ortholog", "direction_all_three", "function", "subsystem", "subsystem_source", "compartment", "glycosylation_role", "glycosylation_role_source", "glycosylation_role_confidence", "evidence_source", "mapping_status", "manual_review_required"]].rename(columns={"direction_all_three": "direction"})
shortlist.to_csv("../data/processed/transcriptomic_shortlist.csv", index=False)
confidence = source_confidence
subsystem_audit = machinery[["record_id", "liu_ao_locus_tag", "yeast_ortholog", "Subsystems or function", "subsystem", "subsystem_source", "subsystem_evidence", "subsystem_candidates", "pathway_order", "manual_review_required"]].copy()
subsystem_audit["assignment_confidence"] = subsystem_audit.subsystem_source.map(confidence)
subsystem_audit = subsystem_audit.rename(columns={"liu_ao_locus_tag": "locus_tag", "Subsystems or function": "liu_subsystem", "subsystem": "final_subsystem", "subsystem_source": "assignment_source", "subsystem_evidence": "evidence_summary", "subsystem_candidates": "conflicting_candidates"})
subsystem_audit.to_csv("../data/processed/qa/subsystem_assignment_audit.csv", index=False)
glyco = machinery_final.loc[machinery_final.glycosylation_role.ne("none"), ["ao_locus_tag", "gene_name", "yeast_ortholog", "function", "glycosylation_role", "glycosylation_role_source", "glycosylation_role_confidence", "subsystem", "compartment"]]
glyco.to_csv("../data/processed/glycosylation_components.csv", index=False)
secondary = {
    "AO090701000141": ["folding"], "AO090003001225": ["folding"],
    "AO090020000468": ["folding"], "AO090009000178": ["er_glycosylation"],
    "AO090003000257": ["erad"], "AO090012000213": ["erad"],
    "AO090005000437": ["snare"], "AO090023000840": ["folding"],
    "AO090003000853": ["erad"],
    "AO090005000718": ["hdsv", "ldsv", "cpy_pathway", "alp_pathway"],
    "AO090701000139": ["snare"], "AO090023000864": ["snare"],
}
primary_rel = machinery_final.loc[machinery_final.subsystem.notna(), ["record_id", "ao_locus_tag", "subsystem", "subsystem_source", "subsystem_confidence", "subsystem_rationale"]].copy()
primary_rel["relationship_type"] = "primary"
secondary_rel = machinery_final[machinery_final.ao_locus_tag.isin(secondary)].loc[:, ["record_id", "ao_locus_tag"]].copy()
secondary_rel["subsystem"] = secondary_rel.ao_locus_tag.map(secondary)
secondary_rel = secondary_rel.explode("subsystem", ignore_index=True)
secondary_rel["subsystem_source"] = "liu2014_composite_or_functional_context"
secondary_rel["subsystem_confidence"] = "medium"
secondary_rel["subsystem_rationale"] = "Secondary role explicitly present in Liu's composite description or supported by reviewed functional context."
secondary_rel["relationship_type"] = "secondary"
gene_subsystems = pd.concat([primary_rel, secondary_rel], ignore_index=True)
gene_subsystems.to_csv("../data/processed/gene_subsystems.csv", index=False)
pathway_nodes = machinery_final[["record_id", "ao_locus_tag", "gene_name", "function", "subsystem", "subsystem_source", "subsystem_confidence", "subsystem_rationale", "pathway_order", "compartment", "sig_all_three", "direction_all_three", "manual_review_required"]].copy()
pathway_nodes["secondary_subsystems"] = pathway_nodes.ao_locus_tag.map(lambda tag: "; ".join(secondary.get(tag, [])) or None)
pathway_nodes["placement_status"] = "assigned"
pathway_nodes.loc[pathway_nodes.subsystem.isna(), "placement_status"] = "unassigned"
pathway_nodes.loc[pathway_nodes.manual_review_required & pathway_nodes.subsystem.notna(), "placement_status"] = "uncertain"
pathway_nodes.to_csv("../data/processed/pathway_nodes.csv", index=False)
flagged = pd.concat([
    machinery_final[machinery_final.mapping_status.isin(["unresolved", "ambiguous", "split", "merged"])],
    clients_final[clients_final.mapping_status.isin(["unresolved", "ambiguous", "split", "merged"])],
], ignore_index=True)
review = pd.concat([machinery_final[machinery_final.manual_review_required], clients_final[clients_final.manual_review_required], clients_final[clients_final.mapping_status.eq("unresolved")]], ignore_index=True).drop_duplicates("record_id")
review = review[["record_id", "record_type", "liu_ao_locus_tag", "gene_name", "yeast_ortholog", "function", "mapping_status", "mapping_method", "subsystem", "subsystem_source", "manual_review_required"]].rename(columns={"liu_ao_locus_tag": "source_locus_tag", "function": "plain_english_function"})
candidate_lookup = uniprot_locus.groupby("ao_locus_tag")["uniprot_accession"].agg(lambda x: "; ".join(sorted(set(x.dropna())))).to_dict()
review["candidate_identifiers"] = review.source_locus_tag.map(candidate_lookup).fillna("none found")
review["issue_type"] = "conflicting subsystem evidence"
review.loc[review.mapping_status.eq("unresolved"), "issue_type"] = "unresolved current identifier"
review.loc[review.mapping_status.eq("ambiguous"), "issue_type"] = "ambiguous current identifier"
review["recommended_action"] = "Review the named evidence; retain unassigned until one interpretation is supported."
review.to_csv("../data/processed/qa/manual_review_queue.csv", index=False)
inspection_ids = ["AO090003000257", "AO090120000461", "AO090001000698", "AO090103000069", "AO090001000733", "AO090012000213"]
extra_ids = machinery_final.loc[~machinery_final.liu_ao_locus_tag.isin(inspection_ids), "liu_ao_locus_tag"].head(14).tolist()
sample = machinery[machinery.liu_ao_locus_tag.isin(inspection_ids + extra_ids)][["liu_ao_locus_tag", "S. cerevisiae ortholog", "Subsystems or function", "Description", "subsystem", "subsystem_source", "subsystem_confidence", "subsystem_rationale", "function", "sig_all_three", "direction_all_three"]].copy()
sample["_order"] = sample.liu_ao_locus_tag.map({tag: i for i, tag in enumerate(inspection_ids + extra_ids)})
sample = sample.sort_values("_order").drop(columns="_order").rename(columns={"liu_ao_locus_tag": "Liu ID", "S. cerevisiae ortholog": "Liu yeast ortholog", "Subsystems or function": "Liu subsystem cell", "Description": "Liu description", "subsystem": "derived subsystem", "subsystem_source": "derived subsystem source", "subsystem_confidence": "derived confidence", "subsystem_rationale": "derived rationale", "function": "derived function"})
assert len(sample) == 20
sample.to_csv("../data/processed/qa/source_fidelity_sample.csv", index=False)
column_descriptions = pd.DataFrame([{"field": field, **metadata} for field, metadata in DATA_DICTIONARY.items()])
column_descriptions.to_csv("../data/processed/column_descriptions.csv", index=False)
dictionary_lines = ["| Field | Definition | Allowed values | Source |", "|---|---|---|---|"]
for row in column_descriptions.itertuples(index=False):
    cells = [str(value).replace("|", "&#124;").replace("\n", " " ) for value in row]
    dictionary_lines.append("| " + " | ".join(cells) + " |")
readme_path = Path("../README.md")
readme = readme_path.read_text(encoding="utf-8")
start, end = "<!-- DATA_DICTIONARY_START -->", "<!-- DATA_DICTIONARY_END -->"
before, remainder = readme.split(start, 1)
_, after = remainder.split(end, 1)
readme_path.write_text(before + start + "\n" + "\n".join(dictionary_lines) + "\n" + end + after, encoding="utf-8")

12481

In [7]:
audit = {
    "machinery_rows": len(machinery_final),
    "client_rows": len(clients_final),
    "machinery_mapping_status": machinery_final.mapping_status.value_counts().to_dict(),
    "client_mapping_status": clients_final.mapping_status.value_counts().to_dict(),
    "machinery_with_uniprot": int(machinery_final.uniprot_accession.notna().sum()),
    "machinery_with_ncbi": int(machinery_final.ncbi_gene_id.notna().sum()),
    "machinery_with_kegg_gene": int(machinery_final.kegg_gene_id.notna().sum()),
    "machinery_with_ko": int(machinery_final.kegg_ko.notna().sum()),
    "kegg_pathway_membership_rows": len(pathway_members),
    "machinery_with_subsystem": int(machinery_final.subsystem.notna().sum()),
    "machinery_subsystem_source": machinery_final.subsystem_source.value_counts().to_dict(),
    "subsystem_conflict_rows": int((
        machinery_final.subsystem_source.eq("unassigned")
        & machinery_final.manual_review_required
        & machinery_final.mapping_status.eq("exact")
    ).sum()),
    "machinery_with_glycosylation_role": int(machinery_final.glycosylation_role.ne("none").sum()),
    "transcriptomic_shortlist_rows": len(shortlist),
    "flagged_rows": len(flagged),
    "important_limitations": [
        "KEGG organism validation now uses the valid HTTPS /list/aor request; KO values remain blank when KEGG supplies no link",
        "Broad KEGG pathway membership is a coarse subsystem proxy and should be reviewed before biological interpretation",
        "A glycosylation_role of none means not assigned in this pass, not experimentally proven absent",
        "Functional evidence is not upgraded by transcriptomic significance",
        "Conflicting UniProt candidates are withheld and flagged rather than selected automatically",
    ],
}
with open("../data/processed/qa/build_audit.json", "w", encoding="utf-8") as handle:
    json.dump(audit, handle, indent=2)
print(json.dumps(audit, indent=2))

{
  "machinery_rows": 369,
  "client_rows": 2269,
  "machinery_mapping_status": {
    "exact": 367,
    "unresolved": 1,
    "ambiguous": 1
  },
  "client_mapping_status": {
    "exact": 2268,
    "unresolved": 1
  },
  "machinery_with_uniprot": 366,
  "machinery_with_ncbi": 284,
  "machinery_with_kegg_gene": 348,
  "machinery_with_ko": 330,
  "kegg_pathway_membership_rows": 196,
  "machinery_with_subsystem": 247,
  "machinery_subsystem_source": {
    "unassigned": 122,
    "liu2014": 108,
    "liu2014_description": 74,
    "yeast_scaffold": 44,
    "kegg_pathway": 13,
    "current_annotation": 3,
    "liu2014_description+uniprot+kegg_ko": 3,
    "curated_liu_composite": 1,
    "aspergillus_homolog+uniprot+yeast_ortholog+kegg_ko": 1
  },
  "subsystem_conflict_rows": 5,
  "machinery_with_glycosylation_role": 62,
  "transcriptomic_shortlist_rows": 51,
  "flagged_rows": 3,
  "important_limitations": [
    "KEGG organism validation now uses the valid HTTPS /list/aor request; KO values rema

## Output: pathway overview and release summary

In [8]:
counts = machinery_final.groupby("subsystem", dropna=False).agg(genes=("record_id", "size"), responsive=("sig_all_three", "sum")).reset_index()
count_lookup = {row.subsystem: (int(row.genes), int(row.responsive)) for row in counts.itertuples()}
unassigned_count = int(machinery_final.subsystem.isna().sum())
unassigned_responsive = int(machinery_final.loc[machinery_final.subsystem.isna(), "sig_all_three"].sum())
labels = list(SUBSYSTEM_ORDER) + ["unassigned / uncertain"]
label_map = {
    "tc": "Translocation (tc)", "dolichol_pathway": "Dolichol-linked glycan assembly",
    "er_glycosylation": "ER N-glycosylation", "folding": "Protein folding",
    "gpi_biosynthesis": "GPI-anchor biosynthesis (gpi)",
    "erad": "ER-associated degradation (erad)",
    "copii": "ER-to-Golgi vesicles (COPII)", "copi": "Golgi-to-ER vesicles (COPI)",
    "golgi_processing": "Golgi glycan processing",
    "ldsv": "Low-density secretory vesicles (ldsv)",
    "hdsv": "High-density secretory vesicles (hdsv)",
    "cpy_pathway": "Vacuolar sorting, CPY route",
    "alp_pathway": "Vacuolar sorting, ALP route",
    "snare": "SNARE vesicle fusion", "septin": "Septin organization",
    "beta_1_6_glucan_biosynthesis": "Beta-1,6-glucan biosynthesis",
    "translation": "Translation",
    "putative_mitochondria_protein": "Putative mitochondrial protein",
    "mitochondrial_m_aaa_protease": "Mitochondrial m-AAA protease",
    "unassigned / uncertain": "Unassigned / uncertain",
}
from datetime import date
import textwrap
build_date = date.today().isoformat()
subtitle_1 = 'Each box is a stage of the secretion pathway. "Responsive" = changed significantly in all three'
subtitle_2 = 'alpha-amylase-overproducing strains (Liu 2014, adj. p < 0.05).'
svg = ["<svg xmlns='http://www.w3.org/2000/svg' width='1100' height='820' viewBox='0 0 1100 820'>", "<rect width='1100' height='820' fill='#fbfaf7'/><text x='50' y='42' font-family='Arial' font-size='26' font-weight='bold'>A. oryzae secretory machinery: primary pathway placement</text>", f"<text x='50' y='70' font-family='Arial' font-size='14' fill='#444'>{subtitle_1}</text><text x='50' y='90' font-family='Arial' font-size='14' fill='#444'>{subtitle_2.replace('<', '&lt;')}</text>"]
for i, label in enumerate(labels):
    col, row = i % 4, i // 4
    x, y = 50 + col * 265, 120 + row * 118
    key = None if label == "unassigned / uncertain" else label
    genes, responsive = (unassigned_count, unassigned_responsive) if key is None else count_lookup.get(key, (0, 0))
    fill = "#f5d8cf" if key is None else "#dbe9e4"
    dash = " stroke-dasharray='7 5'" if key is None else ""
    title_lines = textwrap.wrap(label_map[label], width=29)
    tspans = ''.join(f"<tspan x='{x+14}' dy='{0 if n == 0 else 16}'>{line}</tspan>" for n, line in enumerate(title_lines))
    svg.append(f"<g><title>{label_map[label]}</title><rect x='{x}' y='{y}' width='235' height='92' rx='8' fill='{fill}' stroke='#456' {dash}/><text x='{x+14}' y='{y+21}' font-family='Arial' font-size='13' font-weight='bold'>{tspans}</text><text x='{x+14}' y='{y+70}' font-family='Arial' font-size='14'>{genes} genes</text><text x='{x+126}' y='{y+70}' font-family='Arial' font-size='13' fill='#a34b20'>{responsive} responsive</text></g>")
footer_1 = f"369 machinery genes; {369-unassigned_count} placed; {unassigned_count} unassigned; 51 responsive. Liu et al. 2014, doi:10.1186/1752-0509-8-73. Build {build_date}."
footer_2 = "Low-confidence placements are inferred rather than assigned by Liu; secondary roles are retained in gene_subsystems.csv."
svg.append(f"<text x='50' y='760' font-family='Arial' font-size='12' fill='#555'>{footer_1}</text>")
svg.append(f"<text x='50' y='785' font-family='Arial' font-size='12' fill='#7a4b20'>{footer_2}</text></svg>")
Path("../data/processed/pathway_overview.svg").write_text("".join(svg), encoding="utf-8")
from PIL import Image, ImageDraw, ImageFont
canvas = Image.new("RGB", (1100, 820), "#fbfaf7")
draw = ImageDraw.Draw(canvas)
try:
    title_font, body_font, small_font = (ImageFont.truetype("arial.ttf", size) for size in (26, 15, 12))
except OSError:
    title_font = body_font = small_font = ImageFont.load_default()
draw.text((50, 24), "A. oryzae secretory machinery: primary pathway placement", fill="#223344", font=title_font)
draw.text((50, 62), subtitle_1, fill="#444444", font=small_font)
draw.text((50, 80), subtitle_2, fill="#444444", font=small_font)
for i, label in enumerate(labels):
    col, row = i % 4, i // 4
    x, y = 50 + col * 265, 120 + row * 118
    key = None if label == "unassigned / uncertain" else label
    genes, responsive = (unassigned_count, unassigned_responsive) if key is None else count_lookup.get(key, (0, 0))
    fill = "#f5d8cf" if key is None else "#dbe9e4"
    draw.rounded_rectangle((x, y, x + 235, y + 92), radius=8, fill=fill, outline="#445566", width=2)
    draw.multiline_text((x + 14, y + 10), "\n".join(textwrap.wrap(label_map[label], width=29)), fill="#223344", font=body_font, spacing=2)
    draw.text((x + 14, y + 65), f"{genes} genes", fill="#223344", font=body_font)
    draw.text((x + 126, y + 66), f"{responsive} responsive", fill="#a34b20", font=small_font)
draw.text((50, 748), footer_1, fill="#555555", font=small_font)
draw.text((50, 773), footer_2, fill="#7a4b20", font=small_font)
canvas.save("../data/processed/pathway_overview.png")
assigned = int(machinery_final.subsystem.notna().sum())
ko_coverage = int(machinery_final.kegg_ko.notna().sum())
summary = f"""# Annotation completion summary\n\nLiu supplied subsystem labels for 109 of 369 machinery genes. The 260 blanks still had yeast orthologs; most were additions from A. niger, A. oryzae, reciprocal-best-hit, or InParanoid source lists rather than rows in the original 16-subsystem yeast scaffold. They were therefore unlabeled, not lost.\n\nThis release places **{assigned}/369** genes. Assignments preserve their source in this order: Liu's explicit label, an unambiguous Liu description, the Feizi/Liu yeast scaffold, current UniProt annotation text, then KEGG pathway membership. Conflicts remain visible rather than being silently guessed.\n\nKEGG retrieval now validates the organism through `GET https://rest.kegg.jp/list/aor`. **{ko_coverage}/369** machinery genes have a supplied KO; unavailable values remain blank. KO family mappings add conservative glycosylation roles without replacing Liu-derived roles.\n\nKAR2/BiP is placed primarily in folding, with ERAD retained as a secondary relationship. Seven other genes with explicit cross-subsystem Liu descriptions retain their defensible primary assignment and now expose the additional role in `gene_subsystems.csv`. The pathway figure shows all {len(machinery_final) - assigned} unassigned genes.\n"""
Path("../data/processed/qa/annotation_summary.md").write_text(summary, encoding="utf-8")
print(f"wrote pathway_overview.svg and annotation_summary.md ({assigned}/369 assigned)")

wrote pathway_overview.svg and annotation_summary.md (247/369 assigned)
